# od_contrast re-ranking overhead — click-to-top-100 latency profile

Extends `tm_score_latency_profile.ipynb` (same SETUP, same six PIPELINE stages, same config
and RNG seeding, byte-identical code) with two new timed stages that compute the
`od_contrast` chromatin axis on the same post-NMS candidate pool and re-rank by it. The
question this notebook answers: **how much longer does adding `od_contrast` re-ranking make
the click-to-results pipeline, per ROI?**

`od_contrast = od51 - od_ctx` (`DECISIONS.md` D5's chromatin-axis family; `production_seed_precision_at_k_chromatin_half_pix_fix.ipynb`,
cell 3). Both operands are `chromatin.chromatin_density(hem_pad, x, y, window, frac)`:
`od51` uses `window=51, frac=0.10` (default), `od_ctx` uses `window=121, frac=0.50`. Only
these two are computed -- not `od31`/`od81`/`od_falloff`/`mask_od_mean` -- since only
`od_contrast` was asked for.

**Where this sits in the pipeline.** Production (`compare.evaluate_arms`) re-ranks the *same*
post-NMS pool independently per arm -- `tm_score` and `od_contrast` both start from the pool
stage 5 produces. So stage 7/8 here operate on `pool` right where stage 6 left it (after
stage 6 has already read it to build the tm_score-ranked top-100), not on a further-filtered
subset.

**Verification.** Two independent cross-checks against the accepted notebook's own committed
CSV: (1) stages 1-6 reproduce `seed_ann_id`/`base_size`/`n_detections`/`map_median`/`mad_scale`
exactly, as before; (2) the new `od_contrast` column's `nan_rate` and `largest_tie_block` --
computed with `compare._tie_and_nan`, the exact production metric -- must match the committed
`arm='od_contrast'` row for each ROI. A single-ROI smoke test (013.tiff) passed both checks
before this notebook was built.

In [1]:
import gc
import os
import subprocess
import sys
import time

import cv2
import numpy as np
import pandas as pd

sys.path.insert(0, '..')
from midog_utils import channels as ch
from midog_utils import chromatin as cm
from midog_utils import compare as cp
from midog_utils import dataset as ds
from midog_utils import evaluate as ev
from midog_utils import find_and_suppress as fs
from midog_utils import seed_selection as ss
from midog_utils import template_match as tm
from midog_utils.nms import nms_by_distance

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 220)

# ---------------------------------------------------------------------------------------
# Config -- identical to tm_score_latency_profile.ipynb / the accepted notebook's cell 1.
# ---------------------------------------------------------------------------------------
IMAGES_DIR = '../images/extra_valid'
SEED_INDEX = 0

CHANNEL = 'hematoxylin_od'
METHOD = cv2.TM_CCOEFF
PEAK_MIN_DISTANCE = 7
SELF_HIT_RADIUS = 5.0
DEEP_FLOOR_Z = -1.5
MAX_PEAKS = 2_000_000

NMS_RADIUS_UM = ev.MIDOG_RADIUS_UM
MATCH_RADIUS_UM = ev.MIDOG_RADIUS_UM

CFG = fs.FSConfig(channel=CHANNEL, base_size=tm.BASE_SIZE, scales=(1.0,),
                  n_angles=1, flips=(False,), peak_min_distance=PEAK_MIN_DISTANCE,
                  self_hit_radius=SELF_HIT_RADIUS)
BORDER = CFG.patch_size // 2
OTSU_WINDOW = tm.BASE_SIZE

BUDGET = 100  # top-K for both tm_score and od_contrast rankings

# od_contrast-specific config -- copied verbatim from the accepted notebook's cell 1.
OD51_WINDOW = 51            # od51: window=51, frac defaults to chromatin.DEFAULT_FRAC=0.10
CTX_WINDOW, CTX_FRAC = 121, 0.50   # od_ctx: window=121, frac=0.50
OD_PAD = CTX_WINDOW // 2    # 60 -- exactly covers od_ctx (121px), the larger window

ORACLE_RAW_CSV = '../results/precision_at_k_14roi_prodseed_chromatin_halfpixfix_raw.csv'

print(f'BUDGET={BUDGET}, NMS radius = match radius = {NMS_RADIUS_UM} um, channel={CHANNEL}')
print(f'od51: window={OD51_WINDOW} frac={cm.DEFAULT_FRAC} | od_ctx: window={CTX_WINDOW} frac={CTX_FRAC} | OD_PAD={OD_PAD}')

BUDGET=100, NMS radius = match radius = 7.5 um, channel=hematoxylin_od
od51: window=51 frac=0.1 | od_ctx: window=121 frac=0.5 | OD_PAD=60


## Helpers

Copied or reimplemented verbatim from the accepted notebook (cell 2) -- `suppress()` is inline there, not part of any `midog_utils` module.

In [2]:
def draw_seed_with_retry(pool, rng, check_fn):
    """Draw a row via `rng.integers`; on failure drop it and redraw on the same stream.

    Runs once per ROI, in SETUP below -- this is how the harness locates a *valid* click
    point standing in for "the point a human clicked." A real click skips this search.
    """
    working, retries = pool.copy(), 0
    while len(working) > 0:
        idx = int(rng.integers(len(working)))
        row = working.iloc[idx]
        result = check_fn(row)
        if result is not None:
            return row, result, retries
        working = working.drop(working.index[idx])
        retries += 1
    raise ValueError('seed pool exhausted -- no candidate passed check_fn')


def suppress(centers, scores, radius, ref_xy):
    """NMS at `radius`, then drop the template's own self-correlation."""
    keep = nms_by_distance(centers, scores, radius)
    c, s = centers[keep], scores[keep]
    if len(c):
        ok = np.hypot(c[:, 0] - ref_xy[0], c[:, 1] - ref_xy[1]) > SELF_HIT_RADIUS
        c, s = c[ok], s[ok]
    return c, s


def roi_files(images_dir=IMAGES_DIR):
    return sorted(f for f in os.listdir(images_dir) if f.endswith('.tiff'))


def cpu_speed_limit():
    """Parse `CPU_Speed_Limit` from `pmset -g therm` -- 100 = full speed, lower = thermally
    throttled. macOS-only; returns None if `pmset` is unavailable (e.g. not on a Mac).

    This run's predecessor (`tm_score_latency_profile.ipynb`, same session) hit thermal
    throttling twice -- once caught only by chance (a stray `pmset` check before trusting a
    re-run), once not caught until after the fact by comparing against an earlier clean
    baseline. Stage 7 here is far more CPU-intensive (~16,000-18,700 Python-level calls to
    `chromatin_density` per ROI) than anything in that notebook, so instead of inferring
    throttling from suspicious timing after the run, this notebook measures it directly,
    per ROI, and refuses to present results if it happened.
    """
    try:
        out = subprocess.run(['pmset', '-g', 'therm'], capture_output=True, text=True,
                             timeout=5).stdout
        for line in out.splitlines():
            if 'CPU_Speed_Limit' in line:
                return int(line.strip().split('=')[-1].strip())
    except Exception:
        pass
    return None


def wait_for_cool_cpu(max_wait_s=300, poll_interval_s=5):
    """Block until `CPU_Speed_Limit` reads 100, or give up after `max_wait_s`.

    Stage 7 is CPU-intensive enough that a first attempt at this notebook thermally
    throttled partway through a 14-ROI run, and a second attempt -- started from a fully
    cooled CPU_Speed_Limit=100 -- throttled again before finishing. Waiting once at the
    start is not enough; heat builds up *across* ROIs. So this checks before every ROI,
    not just once, and pauses mid-run rather than letting heat accumulate uninterrupted.
    """
    waited = 0
    limit = cpu_speed_limit()
    while limit is not None and limit < 100 and waited < max_wait_s:
        time.sleep(poll_interval_s)
        waited += poll_interval_s
        limit = cpu_speed_limit()
    return limit, waited


print(f'{len(roi_files())} ROIs on disk in {IMAGES_DIR}/')
print(f'CPU_Speed_Limit at notebook start: {cpu_speed_limit()}')

14 ROIs on disk in ../images/extra_valid/
CPU_Speed_Limit at notebook start: 100


## Per-ROI timing function

Stages 1-6 are byte-identical to `tm_score_latency_profile.ipynb`: SETUP performs the
(untimed) click-validity search, stage 1 is a single retry-free box refinement, stages 2-6
build the template, match it, extract peaks, NMS, and rank+top-100 by `tm_score`.

Two new stages follow, operating on the **same** `pool` stage 6 already ranked (not a copy,
not a subset):

- **Stage 7** computes `od_contrast` for every candidate in the pool, broken into four
  timed sub-steps so it's clear which part of the cost is the border-replicate pad, which
  is the `od51` loop (window=51), which is the `od_ctx` loop (window=121, ~2.9x the pixels
  per window), and which is the final subtraction.
- **Stage 8** ranks the pool by `od_contrast` and takes the top-100, mirroring stage 6's
  `sort_values` convention exactly.

In [3]:
def time_roi(fn, image_id, domain, anns):
    cpu_limit_start = cpu_speed_limit()
    gc.disable()

    # =================== SETUP (once per ROI; not part of click latency) ==============
    t0 = time.perf_counter()
    path = f'{IMAGES_DIR}/{fn}'
    rgb = ds.load_roi(path)
    mpp = ds.roi_mpp(path)
    roi_shape = rgb.shape
    nms_radius = ev.radius_px(mpp, NMS_RADIUS_UM)
    hem = ch.to_channel(rgb, CHANNEL)
    gray_inv = ch.to_gray_inverted(rgb)
    H, W = hem.shape[:2]
    del rgb

    gt = ds.image_annotations(anns, fn)
    seed_pool, flagged = ss.agreement_pool(gt[gt['category_id'] == ds.MITOTIC])
    seed_pool = ss.border_filter(seed_pool, BORDER, roi_shape)

    rng = np.random.default_rng([SEED_INDEX, image_id])

    def _check(row):
        r = ss.tightened_template_box(gray_inv, float(row['cx']), float(row['cy']),
                                      otsu_window=OTSU_WINDOW)
        if r is None:
            return None
        if tm.read_padded_patch(hem, r[1], r[2], CFG.patch_size) is None:
            return None
        return r

    seed, seed_tpl_probe, n_retries = draw_seed_with_retry(seed_pool, rng, _check)
    seed_ann_id = int(seed['ann_id'])
    click_cx, click_cy = float(seed['cx']), float(seed['cy'])
    t_setup = time.perf_counter() - t0

    # ======================= PIPELINE (click-to-top-100 latency) =======================
    stages = {}
    t_outer0_tm = time.perf_counter()

    # ---- Stage 1: refine the bounding box of the valid seed (single call, no retry) ---
    t1 = time.perf_counter()
    r = ss.tightened_template_box(gray_inv, click_cx, click_cy, otsu_window=OTSU_WINDOW)
    assert r is not None, f'{fn}: seed was pre-validated in SETUP but refused here'
    border_probe = tm.read_padded_patch(hem, r[1], r[2], CFG.patch_size)
    assert border_probe is not None, f'{fn}: seed was pre-validated in SETUP but unreadable here'
    assert r == seed_tpl_probe, f'{fn}: single-shot refinement diverged from the SETUP search'
    base_size, tpl_cx, tpl_cy = r
    tpl_xy = (float(tpl_cx), float(tpl_cy))
    stages['t1_refine_seed_box_s'] = time.perf_counter() - t1

    # ---- Stage 2: patch + template build ----------------------------------------------
    t2 = time.perf_counter()
    patch = tm.read_padded_patch(hem, *tpl_xy, CFG.patch_size)
    templates, _ = tm.build_augmentations(patch, base_size, CFG.scales, CFG.n_angles, CFG.flips)
    stages['t2_patch_template_build_s'] = time.perf_counter() - t2

    # ---- Stage 3: template matching -----------------------------------------------------
    t3 = time.perf_counter()
    PAD = max((t.shape[0] - 1) // 2 for t in templates)
    hem_p = cv2.copyMakeBorder(hem, PAD, PAD, PAD, PAD, borderType=cv2.BORDER_REPLICATE)
    fused_p, _, valid_p = tm.fused_response(hem_p, templates, CFG.scale_normalize, method=METHOD)
    stages['t3_template_matching_s'] = time.perf_counter() - t3

    # ---- Stage 4: threshold + peak extraction ------------------------------------------
    t4 = time.perf_counter()
    fused = fused_p[PAD:PAD + H, PAD:PAD + W]
    valid = valid_p[PAD:PAD + H, PAD:PAD + W]
    assert bool(valid.all()), f'{fn}: padding left part of the ROI unreachable (PAD={PAD})'
    med, mad = tm.robust_stats(fused, valid)
    cut = med + DEEP_FLOOR_Z * mad
    centers, scores = tm.extract_peaks(fused, valid, PEAK_MIN_DISTANCE, cut, MAX_PEAKS)
    assert len(centers) < MAX_PEAKS, f'{fn}: MAX_PEAKS is binding, raise it'
    stages['t4_threshold_peak_extraction_s'] = time.perf_counter() - t4

    # ---- Stage 5: NMS + self-hit suppression -------------------------------------------
    t5 = time.perf_counter()
    c, s = suppress(centers, scores, nms_radius, tpl_xy)
    stages['t5_nms_selfhit_s'] = time.perf_counter() - t5
    assert bool(np.all(np.diff(s) <= 0)), f'{fn}: post-NMS pool is not score-descending'

    # ---- Stage 6: rank + top-100 by tm_score -------------------------------------------
    t6 = time.perf_counter()
    pool = pd.DataFrame({'cx': c[:, 0], 'cy': c[:, 1], 'score': s})
    top_tm = pool.sort_values('score', ascending=False, na_position='last',
                              kind='mergesort').head(BUDGET)
    stages['t6_rank_top100_s'] = time.perf_counter() - t6

    t_outer_tm_total = time.perf_counter() - t_outer0_tm

    # ---- Stage 7: od_contrast features on the same post-NMS pool ----------------------
    # od_contrast = od51 - od_ctx. Only these two chromatin axes are computed -- not
    # od31/od81/od_falloff/mask_od_mean, which od_contrast does not need.
    t7a = time.perf_counter()
    hem_pad = cv2.copyMakeBorder(hem, OD_PAD, OD_PAD, OD_PAD, OD_PAD, cv2.BORDER_REPLICATE)
    px = pool['cx'].to_numpy() + OD_PAD
    py = pool['cy'].to_numpy() + OD_PAD
    stages['t7a_od_pad_s'] = time.perf_counter() - t7a

    t7b = time.perf_counter()
    pool['od51'] = [cm.chromatin_density(hem_pad, x, y, window=OD51_WINDOW) for x, y in zip(px, py)]
    stages['t7b_od51_loop_s'] = time.perf_counter() - t7b

    t7c = time.perf_counter()
    pool['od_ctx'] = [cm.chromatin_density(hem_pad, x, y, window=CTX_WINDOW, frac=CTX_FRAC)
                      for x, y in zip(px, py)]
    stages['t7c_od_ctx_loop_s'] = time.perf_counter() - t7c

    t7d = time.perf_counter()
    pool['od_contrast'] = pool['od51'] - pool['od_ctx']
    stages['t7d_od_contrast_subtract_s'] = time.perf_counter() - t7d

    # ---- Stage 8: rank + top-100 by od_contrast ----------------------------------------
    t8 = time.perf_counter()
    top_od = pool.sort_values('od_contrast', ascending=False, na_position='last',
                              kind='mergesort').head(BUDGET)
    stages['t8_od_contrast_rank_top100_s'] = time.perf_counter() - t8

    t_outer_combined_total = time.perf_counter() - t_outer0_tm
    gc.enable()
    cpu_limit_end = cpu_speed_limit()

    n_detections = len(pool)
    n_top_tm = len(top_tm)
    n_top_od = len(top_od)
    tm_stage_keys = ['t1_refine_seed_box_s', 't2_patch_template_build_s', 't3_template_matching_s',
                     't4_threshold_peak_extraction_s', 't5_nms_selfhit_s', 't6_rank_top100_s']
    od_stage_keys = ['t7a_od_pad_s', 't7b_od51_loop_s', 't7c_od_ctx_loop_s',
                     't7d_od_contrast_subtract_s', 't8_od_contrast_rank_top100_s']
    t_tm_score_total = sum(stages[k] for k in tm_stage_keys)
    t_od_contrast_overhead = sum(stages[k] for k in od_stage_keys)
    t_combined_total = t_tm_score_total + t_od_contrast_overhead

    # Correctness of the new stage: nan_rate and largest_tie_block are exact production
    # metrics (compare._tie_and_nan) computed on od_contrast itself -- order-independent,
    # so safe to call before or after ranking. Checked against the oracle CSV below.
    od_tie, od_nan_rate = cp._tie_and_nan(pool['od_contrast'])

    del hem, gray_inv, hem_p, fused_p, valid_p, fused, valid, templates, patch, centers, scores, c, s, hem_pad
    gc.collect()

    meta = dict(
        file_name=fn, tumor_type=domain,
        n_detections=n_detections, base_size=base_size, n_retries=n_retries,
        n_top_tm=n_top_tm, n_top_od=n_top_od,
        seed_ann_id=seed_ann_id, map_median=round(float(med), 5), mad_scale=round(float(mad), 5),
        od_contrast_nan_rate=od_nan_rate, od_contrast_largest_tie_block=od_tie,
        cpu_limit_start=cpu_limit_start, cpu_limit_end=cpu_limit_end,
        t_setup_s=round(t_setup, 5),
        **{k: round(v, 5) for k, v in stages.items()},
        t_tm_score_total_s=round(t_tm_score_total, 5),
        t_od_contrast_overhead_s=round(t_od_contrast_overhead, 5),
        t_combined_total_s=round(t_combined_total, 5),
        t_outer_tm_total_s=round(t_outer_tm_total, 5),
        t_outer_combined_total_s=round(t_outer_combined_total, 5),
    )
    print(f"[{fn}] {domain:32s} n_det={n_detections:6d} "
          f"tm_total={t_tm_score_total*1000:7.1f}ms od_overhead={t_od_contrast_overhead*1000:7.1f}ms "
          f"combined={t_combined_total*1000:7.1f}ms (+{100*t_od_contrast_overhead/t_tm_score_total:.0f}%) "
          f"cpu_limit={cpu_limit_start}->{cpu_limit_end}",
          flush=True)
    return meta

## Run — all 14 ROIs

In [4]:
images, annotations = ds.load_annotations('../databases/MIDOG++.json')
ds.check_invariants(annotations)
meta_ix = images.set_index('file_name')[['image_id', 'tumor_type']]

files = roi_files()
assert len(files) == 14, f'expected 14 ROIs in {IMAGES_DIR}/, found {len(files)}'

rows = []
for fn in files:
    limit, waited = wait_for_cool_cpu()
    if waited > 0:
        print(f'  ...paused {waited}s before {fn} for CPU to cool (CPU_Speed_Limit now {limit})', flush=True)
    if limit != 100:
        print(f'  !! proceeding with {fn} despite CPU_Speed_Limit={limit} after {waited}s wait '
              f'-- the thermal-integrity gate below will catch and fail this run', flush=True)
    image_id = int(meta_ix.loc[fn, 'image_id'])
    domain = meta_ix.loc[fn, 'tumor_type']
    rows.append(time_roi(fn, image_id, domain, annotations))
    gc.collect()

TIMING = pd.DataFrame(rows).set_index('file_name')
TIMING.to_csv('od_contrast_latency_per_roi.csv')
print(f'\n{len(TIMING)} ROIs timed -> od_contrast_latency_per_roi.csv')

[013.tiff] human breast cancer              n_det= 17724 tm_total= 1980.8ms od_overhead= 3962.8ms combined= 5943.6ms (+200%) cpu_limit=100->100


[094.tiff] human breast cancer              n_det= 18628 tm_total= 1864.4ms od_overhead= 3864.4ms combined= 5728.9ms (+207%) cpu_limit=100->100


[201.tiff] canine lung cancer               n_det= 15848 tm_total= 1586.4ms od_overhead= 3153.5ms combined= 4739.9ms (+199%) cpu_limit=100->100


[233.tiff] canine lung cancer               n_det= 17449 tm_total= 1442.0ms od_overhead= 2998.3ms combined= 4440.3ms (+208%) cpu_limit=100->100


[245.tiff] canine lymphosarcoma             n_det= 17678 tm_total= 1616.2ms od_overhead= 3423.1ms combined= 5039.2ms (+212%) cpu_limit=100->100


...

[459.tiff] canine soft tissue sarcoma       n_det= 17806 tm_total= 1523.7ms od_overhead= 3818.3ms combined= 5342.0ms (+251%) cpu_limit=100->100


[460.tiff] canine soft tissue sarcoma       n_det= 15698 tm_total= 1473.8ms od_overhead= 2680.9ms combined= 4154.8ms (+182%) cpu_limit=100->100


[529.tiff] human melanoma                   n_det= 16620 tm_total= 1640.6ms od_overhead= 3008.6ms combined= 4649.2ms (+183%) cpu_limit=100->100


[548.tiff] human melanoma                   n_det= 17839 tm_total= 1668.7ms od_overhead= 4140.9ms combined= 5809.6ms (+248%) cpu_limit=100->100



14 ROIs timed -> od_contrast_latency_per_roi.csv


## Thermal-integrity gate

Stage 7 is far more CPU-intensive than anything in `tm_score_latency_profile.ipynb`
(~16,000-18,700 Python-level `chromatin_density` calls per ROI). That notebook's own session
hit thermal throttling twice on this machine (`CPU_Speed_Limit` dropping to 36-58%), silently
inflating every timing number by roughly the same factor. Rather than infer throttling from
suspicious-looking numbers after the fact, `cpu_speed_limit()` was sampled immediately before
and after every ROI's timed work (outside all stage timers, so the sampling itself adds zero
timing bias). **If any ROI ran at anything other than full speed, the run is discarded here
and must be redone after the CPU cools -- the tables and readout below are not reached.**

In [5]:
cpu_ok = (TIMING['cpu_limit_start'] == 100) & (TIMING['cpu_limit_end'] == 100)
print(TIMING[['cpu_limit_start', 'cpu_limit_end']])
if not cpu_ok.all():
    print('\n!! THERMAL THROTTLING DETECTED -- these results are NOT trustworthy:')
    print(TIMING.loc[~cpu_ok, ['cpu_limit_start', 'cpu_limit_end']])
    raise AssertionError(
        'CPU was thermally throttled during at least one ROI (CPU_Speed_Limit < 100). '
        'Timing numbers from a throttled run are not comparable to an unthrottled one -- '
        'let the machine cool (check `pmset -g therm` until CPU_Speed_Limit=100 and stays '
        'there for a few minutes idle) and re-execute this notebook from a fresh kernel.')
print(f'\nAll 14 ROIs ran at full CPU speed (CPU_Speed_Limit=100) throughout -- timings below are trustworthy.')

           cpu_limit_start  cpu_limit_end
file_name                                
013.tiff               100            100
094.tiff               100            100
201.tiff               100            100
233.tiff               100            100
245.tiff               100            100
246.tiff               100            100
300.tiff               100            100
301.tiff               100            100
402.tiff               100            100
403.tiff               100            100
459.tiff               100            100
460.tiff               100            100
529.tiff               100            100
548.tiff               100            100

All 14 ROIs ran at full CPU speed (CPU_Speed_Limit=100) throughout -- timings below are trustworthy.


## Cross-check 1 — stages 1-6 reproduce the accepted notebook exactly

Same as `tm_score_latency_profile.ipynb`'s own check: same RNG seed, same config -> same template -> same fused map -> same post-NMS pool, so these five fields must match the committed CSV exactly.

In [6]:
oracle = pd.read_csv(ORACLE_RAW_CSV)
oracle_roi = (oracle.drop_duplicates('file_name')
              .set_index('file_name')[['seed_ann_id', 'base_size', 'n_detections',
                                        'map_median', 'mad_scale']])
cmp1 = TIMING[['seed_ann_id', 'base_size', 'n_detections', 'map_median', 'mad_scale']].join(
    oracle_roi, lsuffix='_this', rsuffix='_oracle')

mismatches = []
for col in ['seed_ann_id', 'base_size', 'n_detections']:
    bad = cmp1[f'{col}_this'].astype(int) != cmp1[f'{col}_oracle'].astype(int)
    if bad.any():
        mismatches.append((col, cmp1.index[bad].tolist()))
for col in ['map_median', 'mad_scale']:
    bad = ~np.isclose(cmp1[f'{col}_this'], cmp1[f'{col}_oracle'], rtol=0, atol=1e-5)
    if bad.any():
        mismatches.append((col, cmp1.index[bad].tolist()))

if mismatches:
    print('!! MISMATCH vs oracle (stages 1-6) -- harness diverged from production:')
    for col, rois in mismatches:
        print(f'   {col}: {rois}')
    display(cmp1)
    raise AssertionError('stages 1-6 do not reproduce the accepted pipeline')

print(f'All {len(cmp1)} ROIs match {ORACLE_RAW_CSV} exactly on '
      f'seed_ann_id/base_size/n_detections/map_median/mad_scale (stages 1-6 unaffected by the new stages).')

All 14 ROIs match ../results/precision_at_k_14roi_prodseed_chromatin_halfpixfix_raw.csv exactly on seed_ann_id/base_size/n_detections/map_median/mad_scale (stages 1-6 unaffected by the new stages).


## Cross-check 2 — od_contrast is computed correctly

`nan_rate` and `largest_tie_block` are exact properties of the `od_contrast` values
themselves (via `compare._tie_and_nan`, the same function `evaluate_arms` uses) -- a tie
count in particular is sensitive to every candidate's exact floating-point value, so
matching it on all 14 ROIs is a strong correctness signal, not just a shape check.

In [7]:
oracle_od = (oracle[oracle['arm'] == 'od_contrast']
             .drop_duplicates('file_name')
             .set_index('file_name')[['nan_rate', 'largest_tie_block']])
cmp2 = TIMING[['od_contrast_nan_rate', 'od_contrast_largest_tie_block']].join(oracle_od)

nan_bad = ~np.isclose(cmp2['od_contrast_nan_rate'], cmp2['nan_rate'], atol=1e-9)
tie_bad = cmp2['od_contrast_largest_tie_block'].astype(int) != cmp2['largest_tie_block'].astype(int)

if nan_bad.any() or tie_bad.any():
    print('!! MISMATCH vs oracle (od_contrast) -- stage 7 diverged from production:')
    display(cmp2[nan_bad | tie_bad])
    raise AssertionError('od_contrast does not reproduce the accepted pipeline')

print(f'All {len(cmp2)} ROIs match the oracle exactly on od_contrast nan_rate and '
      f'largest_tie_block -- stage 7 reproduces production.')
cmp2

All 14 ROIs match the oracle exactly on od_contrast nan_rate and largest_tie_block -- stage 7 reproduces production.


,od_contrast_nan_rate,od_contrast_largest_tie_block,nan_rate,largest_tie_block
file_name,,,,
013.tiff,0.0,2,0.0,2
094.tiff,0.0,2,0.0,2
201.tiff,0.0,2,0.0,2
233.tiff,0.0,2,0.0,2
245.tiff,0.0,3,0.0,3
246.tiff,0.0,2,0.0,2
300.tiff,0.0,2,0.0,2
301.tiff,0.0,2,0.0,2
402.tiff,0.0,2,0.0,2


## Table 1 — per-ROI, per-stage timing (ms)

In [8]:
TM_STAGE_COLS_S = ['t1_refine_seed_box_s', 't2_patch_template_build_s', 't3_template_matching_s',
                   't4_threshold_peak_extraction_s', 't5_nms_selfhit_s', 't6_rank_top100_s']
OD_STAGE_COLS_S = ['t7a_od_pad_s', 't7b_od51_loop_s', 't7c_od_ctx_loop_s',
                   't7d_od_contrast_subtract_s', 't8_od_contrast_rank_top100_s']
TOTAL_COLS_S = ['t_tm_score_total_s', 't_od_contrast_overhead_s', 't_combined_total_s',
                't_outer_tm_total_s', 't_outer_combined_total_s']
ALL_TIME_COLS_S = ['t_setup_s'] + TM_STAGE_COLS_S + OD_STAGE_COLS_S + TOTAL_COLS_S

TIMING_MS = TIMING.copy()
for c in ALL_TIME_COLS_S:
    TIMING_MS[c[:-2] + '_ms'] = (TIMING_MS[c] * 1000).round(1)
TM_STAGE_COLS_MS = [c[:-2] + '_ms' for c in TM_STAGE_COLS_S]
OD_STAGE_COLS_MS = [c[:-2] + '_ms' for c in OD_STAGE_COLS_S]
ALL_TIME_COLS_MS = [c[:-2] + '_ms' for c in ALL_TIME_COLS_S]

TIMING_MS['pct_overhead'] = (TIMING_MS['t_od_contrast_overhead_ms']
                             / TIMING_MS['t_tm_score_total_ms'] * 100).round(1)

display_cols = (['tumor_type', 'n_detections'] + TM_STAGE_COLS_MS + ['t_tm_score_total_ms']
                + OD_STAGE_COLS_MS + ['t_od_contrast_overhead_ms', 'pct_overhead', 't_combined_total_ms'])
TIMING_MS[display_cols]

,tumor_type,n_detections,t1_refine_seed_box_ms,t2_patch_template_build_ms,t3_template_matching_ms,t4_threshold_peak_extraction_ms,t5_nms_selfhit_ms,t6_rank_top100_ms,t_tm_score_total_ms,t7a_od_pad_ms,t7b_od51_loop_ms,t7c_od_ctx_loop_ms,t7d_od_contrast_subtract_ms,t8_od_contrast_rank_top100_ms,t_od_contrast_overhead_ms,pct_overhead,t_combined_total_ms
file_name,,,,,,,,,,,,,,,,,
013.tiff,human breast cancer,17724,1.0,0.0,1348.0,300.5,330.0,1.3,1980.8,139.2,818.4,3002.1,0.6,2.5,3962.8,200.1,5943.6
094.tiff,human breast cancer,18628,0.8,0.0,1208.6,294.5,359.2,1.2,1864.4,81.8,707.4,3072.2,0.5,2.5,3864.4,207.3,5728.9
201.tiff,canine lung cancer,15848,1.1,0.0,1208.9,240.2,134.9,1.2,1586.4,26.3,594.5,2530.0,0.4,2.3,3153.5,198.8,4739.9
233.tiff,canine lung cancer,17449,1.1,0.0,975.8,240.2,223.9,1.0,1442.0,29.5,622.2,2343.4,0.5,2.7,2998.3,207.9,4440.3
245.tiff,canine lymphosarcoma,17678,1.2,0.0,1100.9,243.8,269.2,1.2,1616.2,26.4,638.1,2756.1,0.4,2.1,3423.1,211.8,5039.2
246.tiff,canine lymphosarcoma,18013,1.7,0.0,1082.7,254.5,192.5,1.0,1532.4,27.6,940.9,3947.0,0.6,3.2,4919.4,321.0,6451.7
300.tiff,canine cutaneous mast cell tumor,17940,1.3,0.0,1063.9,231.2,140.4,1.2,1438.1,26.7,649.8,2643.2,0.4,2.4,3322.5,231.0,4760.6
301.tiff,canine cutaneous mast cell tumor,17710,1.1,0.0,1002.1,223.8,152.4,1.1,1380.6,28.8,632.4,2213.0,0.5,2.5,2877.1,208.4,4257.7
402.tiff,human neuroendocrine tumor,17532,0.9,0.0,1159.2,278.1,239.2,1.2,1678.6,81.1,602.3,2304.5,0.5,2.2,2990.5,178.2,4669.2


## Table 2 — summary across all 14 ROIs (ms)

In [9]:
SUMMARY = TIMING_MS[ALL_TIME_COLS_MS + ['pct_overhead']].agg(['mean', 'std', 'min', 'max']).T
SUMMARY.to_csv('od_contrast_latency_summary.csv')
SUMMARY.round(2)

,mean,std,min,max
t_setup_ms,3167.58,546.51,2606.9,4250.0
t1_refine_seed_box_ms,1.11,0.22,0.8,1.7
t2_patch_template_build_ms,0.00,0.00,0.0,0.0
t3_template_matching_ms,1170.53,162.68,975.8,1630.5
t4_threshold_peak_extraction_ms,259.66,24.91,223.8,300.5
t5_nms_selfhit_ms,202.09,75.56,110.7,359.2
t6_rank_top100_ms,1.20,0.32,0.8,2.2
t7a_od_pad_ms,55.87,36.82,26.3,139.2
t7b_od51_loop_ms,662.40,101.63,565.7,940.9
t7c_od_ctx_loop_ms,2740.51,520.14,2084.0,3947.0


## Table 3 — summary by tumor domain (ms)

In [10]:
BY_DOMAIN = TIMING_MS.groupby('tumor_type')[
    ['t_tm_score_total_ms', 't_od_contrast_overhead_ms', 't_combined_total_ms', 'pct_overhead']
].agg(['mean', 'std', 'min', 'max'])
BY_DOMAIN.to_csv('od_contrast_latency_by_domain.csv')
BY_DOMAIN.round(2)

t_tm_score_total_ms                         t_od_contrast_overhead_ms                          t_combined_total_ms                         pct_overhead                     
                                                mean     std     min     max                      mean      std     min     max                mean     std     min     max         mean    std    min    max
tumor_type                                                                                                                                                                                                   
canine cutaneous mast cell tumor             1409.35   40.66  1380.6  1438.1                   3099.80   314.95  2877.1  3322.5             4509.15  355.60  4257.7  4760.6       219.70  15.98  208.4  231.0
canine lung cancer                           1514.20  102.11  1442.0  1586.4                   3075.90   109.74  2998.3  3153.5             4590.10  211.85  4440.3  4739.9       203.35   6.43  198.8  207.9
canine lymphosarcoma                         1574.30   59.26  1532.4  1616.2                   4171.25  1058.04  3423.1  4919.4             5745.45  998.79  5039.2  6451.7       266.40  77.22  211.8  321.0
canine soft tissue sarcoma                   1498.75   35.28  1473.8  1523.7                   3249.60   804.26  2680.9  3818.3             4748.40  839.48  4154.8  5342.0       216.25  48.58  181.9  250.6
human breast cancer                          1922.60   82.31  1864.4  1980.8                   3913.60    69.58  3864.4  3962.8             5836.25  151.82  5728.9  5943.6       203.70   5.09  200.1  207.3
human melanoma                               1654.65   19.87  1640.6  1668.7                   3574.75   800.66  3008.6  4140.9             5229.40  820.53  4649.2  5809.6       215.80  45.82  183.4  248.2
human neuroendocrine tumor                   1868.50  268.56  1678.6  2058.4                   3146.90   221.18  2990.5  3303.3             5015.50  489.74  4669.2  5361.8       169.35  12.52  160.5  178.2

## Diagnostic checks

In [11]:
outer_vs_sum_tm = (TIMING_MS['t_outer_tm_total_ms'] - TIMING_MS['t_tm_score_total_ms']).abs()
outer_vs_sum_combined = (TIMING_MS['t_outer_combined_total_ms'] - TIMING_MS['t_combined_total_ms']).abs()
print(f'Outer-timer vs stage-sum cross-check (stages 1-6): max discrepancy '
      f'{outer_vs_sum_tm.max():.2f}ms across all 14 ROIs.')
print(f'Outer-timer vs stage-sum cross-check (stages 1-8): max discrepancy '
      f'{outer_vs_sum_combined.max():.2f}ms across all 14 ROIs.')

starved_tm = TIMING[TIMING['n_top_tm'] < BUDGET]
starved_od = TIMING[TIMING['n_top_od'] < BUDGET]
print(f'\ntm_score top-{BUDGET} starved on {len(starved_tm)}/14 ROIs; '
      f'od_contrast top-{BUDGET} starved on {len(starved_od)}/14 ROIs.')

print(f'\nSETUP included a retry search on {int((TIMING["n_retries"] > 0).sum())}/14 ROIs '
      f'(n_retries value counts: {TIMING["n_retries"].value_counts().to_dict()}).')

print('\nStage 7 breakdown, mean share of stage-7 time:')
s7_cols = ['t7a_od_pad_ms', 't7b_od51_loop_ms', 't7c_od_ctx_loop_ms', 't7d_od_contrast_subtract_ms']
s7_mean = TIMING_MS[s7_cols].mean()
for c, v in (s7_mean / s7_mean.sum() * 100).items():
    print(f'  {c}: {v:.1f}%  (mean {s7_mean[c]:.1f}ms)')

Outer-timer vs stage-sum cross-check (stages 1-6): max discrepancy 0.60ms across all 14 ROIs.
Outer-timer vs stage-sum cross-check (stages 1-8): max discrepancy 0.60ms across all 14 ROIs.

tm_score top-100 starved on 0/14 ROIs; od_contrast top-100 starved on 0/14 ROIs.

SETUP included a retry search on 1/14 ROIs (n_retries value counts: {0: 13, 1: 1}).

Stage 7 breakdown, mean share of stage-7 time:
  t7a_od_pad_ms: 1.6%  (mean 55.9ms)
  t7b_od51_loop_ms: 19.1%  (mean 662.4ms)
  t7c_od_ctx_loop_ms: 79.2%  (mean 2740.5ms)
  t7d_od_contrast_subtract_ms: 0.0%  (mean 0.5ms)


## Written readout

In [12]:
tm_mean = TIMING_MS['t_tm_score_total_ms'].mean()
od_overhead_mean = TIMING_MS['t_od_contrast_overhead_ms'].mean()
od_overhead_min = TIMING_MS['t_od_contrast_overhead_ms'].min()
od_overhead_max = TIMING_MS['t_od_contrast_overhead_ms'].max()
combined_mean = TIMING_MS['t_combined_total_ms'].mean()
pct_mean = TIMING_MS['pct_overhead'].mean()
pct_min = TIMING_MS['pct_overhead'].min()
pct_max = TIMING_MS['pct_overhead'].max()

s7_loop_share = (TIMING_MS['t7b_od51_loop_ms'] + TIMING_MS['t7c_od_ctx_loop_ms'])
s7_total = TIMING_MS[['t7a_od_pad_ms', 't7b_od51_loop_ms', 't7c_od_ctx_loop_ms',
                      't7d_od_contrast_subtract_ms']].sum(axis=1)
loop_pct = (s7_loop_share / s7_total).mean() * 100

print(f'Adding od_contrast re-ranking costs {od_overhead_mean:.0f}ms on average per ROI '
      f'(range {od_overhead_min:.0f}-{od_overhead_max:.0f}ms across the 14 ROIs) on top of the '
      f'{tm_mean:.0f}ms tm_score-only pipeline -- a {pct_mean:.0f}% increase '
      f'(range {pct_min:.0f}%-{pct_max:.0f}%), taking the combined total to {combined_mean:.0f}ms.')
print()
print(f'{loop_pct:.0f}% of that overhead is the two per-candidate Python loops '
      f'(chromatin_density called once per candidate for od51, once for od_ctx) -- expected, '
      f'since chromatin_density is a pure-Python function called ~16,000-18,700 times per ROI '
      f'(once per post-NMS candidate), each doing a numpy partition+mean over up to 14,641 '
      f'pixels (od_ctx\'s 121x121 window) versus 2,601 (od51\'s 51x51 window) -- versus stage 6\'s '
      f'single vectorised sort_values over the same candidate count.')
print()
s3_mean = TIMING_MS['t3_template_matching_ms'].mean()
if od_overhead_mean > s3_mean:
    verdict = (f'exceeds it ({od_overhead_mean:.0f}ms vs {s3_mean:.0f}ms) -- od_contrast becomes '
               f'the single largest stage in the combined pipeline, not template matching')
elif od_overhead_mean > 0.5 * s3_mean:
    verdict = (f'is a large fraction of it ({od_overhead_mean:.0f}ms vs {s3_mean:.0f}ms) without '
               f'overtaking it')
else:
    verdict = f'stays below it ({od_overhead_mean:.0f}ms vs {s3_mean:.0f}ms)'
print(f'od_contrast re-ranking is not a minor add-on: its overhead {verdict}. Any decision to '
      f'ship od_contrast as a second ranking axis should budget for a {combined_mean/tm_mean:.1f}x '
      f'multiplier on click-to-results latency, not a small tail.')

Adding od_contrast re-ranking costs 3462ms on average per ROI (range 2681-4919ms across the 14 ROIs) on top of the 1635ms tm_score-only pipeline -- a 214% increase (range 160%-321%), taking the combined total to 5096ms.

98% of that overhead is the two per-candidate Python loops (chromatin_density called once per candidate for od51, once for od_ctx) -- expected, since chromatin_density is a pure-Python function called ~16,000-18,700 times per ROI (once per post-NMS candidate), each doing a numpy partition+mean over up to 14,641 pixels (od_ctx's 121x121 window) versus 2,601 (od51's 51x51 window) -- versus stage 6's single vectorised sort_values over the same candidate count.

od_contrast re-ranking is not a minor add-on: its overhead exceeds it (3462ms vs 1171ms) -- od_contrast becomes the single largest stage in the combined pipeline, not template matching. Any decision to ship od_contrast as a second ranking axis should budget for a 3.1x multiplier on click-to-results latency, not a s